<a href="https://colab.research.google.com/github/zavisk/AutonomousMedicalImage_TriageAgent/blob/main/rag_agentic_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Autonomous Medical Image Triage Agent
## Part 2: RAG Pipeline + Agentic Workflow
### Uses ensemble of 4 trained models from Part 1

## 1. Install Dependencies

In [ ]:
!pip install chromadb sentence-transformers langgraph langchain langchain-core -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 1.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the so

## 2. Imports

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
from datetime import datetime
from typing import TypedDict, List, Dict, Optional
from pathlib import Path

import chromadb
from sentence_transformers import SentenceTransformer
from langgraph.graph import StateGraph, END

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: cpu


## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MODEL_DIR = '/content/drive/MyDrive/chexpert_outputs/'
DATA_DIR = '/content/chexpert/CheXpert-v1.0-small/'

TARGET_LABELS = ['Cardiomegaly', 'Pneumonia', 'Pneumothorax', 'Edema', 'Pleural Effusion']
IMG_SIZE = 256

print(f"Model directory: {MODEL_DIR}")
print(f"Files in Drive: {os.listdir(MODEL_DIR)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model directory: /content/drive/MyDrive/chexpert_outputs/
Files in Drive: ['label_distribution.png', 'sample_images.png', 'densenet_121_best.pth', 'efficientnet_b0_best.pth', 'resnet_50_best.pth', 'efficientnet_b3_best.pth', 'per_class_all_models.png', 'training_curves.png', 'model_comparison.png', 'model_comparison.csv', 'config.json', 'training_history.json']


## 4. Model Classes

In [ ]:
class DenseNet121(nn.Module):
    """DenseNet-121: Dense connections for efficient feature reuse."""
    def __init__(self, num_classes=5):
        super(DenseNet121, self).__init__()
        self.model = models.densenet121(weights='IMAGENET1K_V1')
        in_features = self.model.classifier.in_features
        self.model.classifier = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(512, num_classes)
        )
    def forward(self, x):
        return self.model(x)

class EfficientNetB0(nn.Module):
    """EfficientNet-B0: Compound scaling for balanced depth/width/resolution."""
    def __init__(self, num_classes=5):
        super(EfficientNetB0, self).__init__()
        self.model = models.efficientnet_b0(weights='IMAGENET1K_V1')
        in_features = self.model.classifier[1].in_features
        self.model.classifier = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(512, num_classes)
        )
    def forward(self, x):
        return self.model(x)

class ResNet50(nn.Module):
    """ResNet-50: Residual connections to solve vanishing gradient problem."""
    def __init__(self, num_classes=5):
        super(ResNet50, self).__init__()
        self.model = models.resnet50(weights='IMAGENET1K_V1')
        in_features = self.model.fc.in_features
        self.model.fc = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(512, num_classes)
        )
    def forward(self, x):
        return self.model(x)

class EfficientNetB3(nn.Module):
    """EfficientNet-B3: Larger compound scaled model for better accuracy than B0."""
    def __init__(self, num_classes=5):
        super(EfficientNetB3, self).__init__()
        self.model = models.efficientnet_b3(weights='IMAGENET1K_V1')
        in_features = self.model.classifier[1].in_features
        self.model.classifier = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(512, num_classes)
        )
    def forward(self, x):
        return self.model(x)

MODELS = {
    'DenseNet-121':    DenseNet121,
    'EfficientNet-B0': EfficientNetB0,
    'ResNet-50':       ResNet50,
    'EfficientNet-B3': EfficientNetB3,
}

print("Model classes defined! ✅")

Model classes defined! ✅


## 5. Load Ensemble Models

In [ ]:
def load_model(model_name, ModelClass, model_dir, device):
    """Load a saved model checkpoint."""
    model = ModelClass(num_classes=len(TARGET_LABELS))
    ckpt_path = os.path.join(model_dir, f"{model_name.lower().replace('-', '_')}_best.pth")
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"✅ {model_name} — Best AUC: {checkpoint.get('best_auc', 'N/A')}")
    else:
        model.load_state_dict(checkpoint)
        print(f"✅ {model_name} — loaded")
    model = model.to(device)
    model.eval()
    return model

# Load all 4 models
loaded_models = {}
for model_name, ModelClass in MODELS.items():
    loaded_models[model_name] = load_model(model_name, ModelClass, MODEL_DIR, DEVICE)

print(f"\n{len(loaded_models)} models loaded for ensemble! ✅")

✅ DenseNet-121 — Best AUC: 0.8607874015370818
✅ EfficientNet-B0 — Best AUC: 0.8782616494544424
✅ ResNet-50 — Best AUC: 0.8670397334591213
✅ EfficientNet-B3 — Best AUC: 0.874674724828919

4 models loaded for ensemble! ✅


## 6. Image Transforms & Ensemble Prediction

In [ ]:
val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def predict_ensemble(image_path, models_dict, transform, labels, device, threshold=0.5):
    """Run ensemble inference on a single chest X-ray image."""
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)

    all_probs = []
    with torch.no_grad():
        for model_name, model in models_dict.items():
            outputs = torch.sigmoid(model(image_tensor))
            all_probs.append(outputs.cpu().numpy()[0])

    # Average predictions across all models
    avg_probs = np.mean(all_probs, axis=0)
    predictions = {label: float(prob) for label, prob in zip(labels, avg_probs)}
    positive_findings = [label for label, prob in predictions.items() if prob >= threshold]

    print("\nEnsemble Prediction Results")
    print("-" * 40)
    for label, prob in predictions.items():
        status = "POSITIVE" if prob >= threshold else "Negative"
        indicator = "⚠️" if prob >= threshold else "✓"
        print(f"{indicator} {label}: {prob:.1%} ({status})")

    return predictions, positive_findings

print("Transforms and ensemble predictor ready! ✅")

Transforms and ensemble predictor ready! ✅


## 7. RAG Pipeline

In [ ]:
class MedicalRAGPipeline:
    """RAG pipeline using ChromaDB + SentenceTransformers for medical evidence retrieval."""

    KNOWLEDGE_BASE = [
        {"id": "ptx_1", "text": "Pneumothorax is the presence of air in the pleural space. Tension pneumothorax is life-threatening and requires immediate needle decompression. Treatment: chest tube insertion, oxygen therapy.", "category": "Pneumothorax"},
        {"id": "ptx_2", "text": "Pneumothorax urgency: large pneumothorax (>20%) or tension pneumothorax is CRITICAL requiring emergency intervention within 15 minutes. Small pneumothorax may be monitored.", "category": "Pneumothorax"},
        {"id": "edema_1", "text": "Pulmonary edema is fluid accumulation in the lungs. Acute pulmonary edema is a medical emergency. Treatment: diuretics (furosemide), oxygen, positioning, vasodilators.", "category": "Edema"},
        {"id": "edema_2", "text": "Pulmonary edema urgency: acute onset with respiratory distress is CRITICAL. Chronic stable edema is MODERATE. Requires immediate cardiology or ICU consultation for acute cases.", "category": "Edema"},
        {"id": "pna_1", "text": "Pneumonia is an infection causing lung inflammation. Community-acquired pneumonia (CAP) treated with antibiotics. Severe pneumonia with sepsis requires ICU admission.", "category": "Pneumonia"},
        {"id": "pna_2", "text": "Pneumonia urgency: severe pneumonia with hypoxia or sepsis is URGENT requiring immediate antibiotics and possible ICU. Mild community pneumonia is MODERATE with outpatient treatment.", "category": "Pneumonia"},
        {"id": "cardio_1", "text": "Cardiomegaly (enlarged heart) indicates underlying cardiac disease such as heart failure, cardiomyopathy, or valve disease. Requires echocardiogram and cardiology consultation.", "category": "Cardiomegaly"},
        {"id": "cardio_2", "text": "Cardiomegaly urgency: new onset with symptoms is URGENT. Chronic known cardiomegaly is MODERATE requiring cardiology follow-up and optimization of heart failure therapy.", "category": "Cardiomegaly"},
        {"id": "pe_1", "text": "Pleural effusion is fluid in the pleural space. Causes include heart failure, infection, malignancy. Large effusions causing respiratory compromise require thoracentesis.", "category": "Pleural Effusion"},
        {"id": "pe_2", "text": "Pleural effusion urgency: large effusion with respiratory distress is URGENT. Small effusion without symptoms is MODERATE requiring outpatient investigation for underlying cause.", "category": "Pleural Effusion"},
        {"id": "urgency_1", "text": "CRITICAL: Immediate life-threatening condition requiring intervention within 15 minutes. Page on-call specialist immediately. Examples: tension pneumothorax, acute pulmonary edema.", "category": "Urgency"},
        {"id": "urgency_2", "text": "URGENT: Serious condition requiring evaluation within 1 hour. Notify specialist promptly. MODERATE: Requires same-day evaluation. ROUTINE: Schedule follow-up within 1 week.", "category": "Urgency"},
    ]

    def __init__(self, persist_dir=None):
        print("Initializing RAG pipeline...")
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')
        if persist_dir:
            self.client = chromadb.PersistentClient(path=persist_dir)
        else:
            self.client = chromadb.Client()
        self.collection = self.client.get_or_create_collection(
            name="medical_knowledge",
            metadata={"hnsw:space": "cosine"}
        )
        if self.collection.count() == 0:
            self._index_documents()
        else:
            print(f"Knowledge base already indexed ({self.collection.count()} docs) ✅")

    def _index_documents(self):
        texts = [doc['text'] for doc in self.KNOWLEDGE_BASE]
        embeddings = self.embedder.encode(texts).tolist()
        self.collection.add(
            ids=[doc['id'] for doc in self.KNOWLEDGE_BASE],
            embeddings=embeddings,
            documents=texts,
            metadatas=[{'category': doc['category']} for doc in self.KNOWLEDGE_BASE]
        )
        print(f"Indexed {len(self.KNOWLEDGE_BASE)} medical documents ✅")

    def get_evidence_for_findings(self, positive_findings, n_results=2):
        """Retrieve clinical evidence for each positive finding."""
        if not positive_findings:
            return "No positive findings detected. Patient appears normal."
        all_evidence = []
        for finding in positive_findings:
            query = f"{finding} clinical guidelines treatment urgency"
            query_embedding = self.embedder.encode([query]).tolist()
            results = self.collection.query(
                query_embeddings=query_embedding,
                n_results=n_results
            )
            all_evidence.extend(results['documents'][0])
        # Add urgency guidelines
        urgency_embedding = self.embedder.encode(["urgency triage levels critical urgent moderate"]).tolist()
        urgency_results = self.collection.query(query_embeddings=urgency_embedding, n_results=1)
        all_evidence.extend(urgency_results['documents'][0])
        return "\n\n".join(all_evidence[:5])

rag_pipeline = MedicalRAGPipeline()
print("RAG Pipeline ready! ✅")

Initializing RAG pipeline...


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Knowledge base already indexed (12 docs) ✅
RAG Pipeline ready! ✅


## 8. Agentic Workflow (LangGraph)

In [ ]:
class TriageState(TypedDict):
    """State object passed between all agent nodes."""
    image_path:      str
    patient_id:      str
    predictions:     Dict[str, float]
    positive_findings: List[str]
    evidence:        str
    urgency:         str
    urgency_reason:  str
    specialist:      str
    routing_reason:  str
    report:          str
    alerts:          List[str]
    timestamp:       str


# ── Node 1: Analyze Image ──────────────────────────────────────────────────────
def analyze_image(state: TriageState) -> TriageState:
    """Run ensemble model on the X-ray image."""
    print(f"[1/6] Analyzing image for patient {state['patient_id']}...")
    predictions, positive_findings = predict_ensemble(
        state['image_path'], loaded_models, val_transforms, TARGET_LABELS, DEVICE
    )
    state['predictions'] = predictions
    state['positive_findings'] = positive_findings
    state['timestamp'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    return state


# ── Node 2: Retrieve Evidence ──────────────────────────────────────────────────
def retrieve_evidence(state: TriageState) -> TriageState:
    """Retrieve clinical guidelines from RAG knowledge base."""
    print(f"[2/6] Retrieving clinical evidence for: {state['positive_findings']}")
    state['evidence'] = rag_pipeline.get_evidence_for_findings(state['positive_findings'])
    return state


# ── Node 3: Classify Urgency ───────────────────────────────────────────────────
def classify_urgency(state: TriageState) -> TriageState:
    """Rule-based urgency classification based on findings and confidence."""
    print("[3/6] Classifying urgency...")
    preds = state['predictions']
    urgency = "ROUTINE"
    reason = "No significant findings detected."

    if preds.get('Pneumothorax', 0) > 0.7:
        urgency = "CRITICAL"
        reason = f"High confidence Pneumothorax ({preds['Pneumothorax']:.1%}) — possible tension pneumothorax."
    elif preds.get('Edema', 0) > 0.7:
        urgency = "CRITICAL"
        reason = f"High confidence Pulmonary Edema ({preds['Edema']:.1%}) — acute respiratory compromise."
    elif preds.get('Pneumonia', 0) > 0.6:
        urgency = "URGENT"
        reason = f"Pneumonia detected ({preds['Pneumonia']:.1%}) — requires prompt antibiotic therapy."
    elif preds.get('Pneumothorax', 0) > 0.5:
        urgency = "URGENT"
        reason = f"Moderate confidence Pneumothorax ({preds['Pneumothorax']:.1%}) — requires monitoring."
    elif preds.get('Cardiomegaly', 0) > 0.5 or preds.get('Pleural Effusion', 0) > 0.5:
        urgency = "MODERATE"
        reason = "Cardiomegaly or Pleural Effusion detected — requires specialist evaluation."
    elif state['positive_findings']:
        urgency = "MODERATE"
        reason = f"Findings detected: {', '.join(state['positive_findings'])}."

    state['urgency'] = urgency
    state['urgency_reason'] = reason
    print(f"    Urgency: {urgency}")
    return state


# ── Node 4: Determine Routing ──────────────────────────────────────────────────
def determine_routing(state: TriageState) -> TriageState:
    """Assign specialist based on findings and urgency."""
    print("[4/6] Determining specialist routing...")
    preds = state['predictions']
    specialist = "General Medicine"
    reason = "Default routing for general evaluation."

    if state['urgency'] == "CRITICAL":
        specialist = "Emergency Medicine + ICU"
        reason = "Critical condition requires immediate emergency intervention."
    elif preds.get('Pneumothorax', 0) > 0.5:
        specialist = "Thoracic Surgery"
        reason = "Pneumothorax requires thoracic surgery evaluation."
    elif preds.get('Edema', 0) > 0.5:
        specialist = "Cardiology"
        reason = "Pulmonary edema likely cardiac in origin."
    elif preds.get('Pneumonia', 0) > 0.5:
        specialist = "Pulmonology / Infectious Disease"
        reason = "Pneumonia requires pulmonology or infectious disease management."
    elif preds.get('Cardiomegaly', 0) > 0.5:
        specialist = "Cardiology"
        reason = "Cardiomegaly requires cardiac workup and echocardiogram."
    elif preds.get('Pleural Effusion', 0) > 0.5:
        specialist = "Pulmonology"
        reason = "Pleural effusion requires investigation and possible thoracentesis."

    state['specialist'] = specialist
    state['routing_reason'] = reason
    print(f"    Routing to: {specialist}")
    return state


# ── Node 5: Generate Report ────────────────────────────────────────────────────
def generate_report(state: TriageState) -> TriageState:
    """Generate structured clinical triage report."""
    print("[5/6] Generating triage report...")
    findings_str = "\n".join(
        [f"  • {label}: {prob:.1%}" for label, prob in state['predictions'].items()]
    )
    positive_str = ", ".join(state['positive_findings']) if state['positive_findings'] else "None"
    evidence_preview = state['evidence'][:800] + "..." if len(state['evidence']) > 800 else state['evidence']

    report = f"""
{'='*60}
MEDICAL TRIAGE REPORT
{'='*60}
Patient ID  : {state['patient_id']}
Timestamp   : {state['timestamp']}
Model       : Ensemble (DenseNet-121, EfficientNet-B0, ResNet-50, EfficientNet-B3)

── FINDINGS ────────────────────────────────────────────────
{findings_str}

Positive Findings : {positive_str}

── TRIAGE DECISION ─────────────────────────────────────────
Urgency Level : {state['urgency']}
Reason        : {state['urgency_reason']}

── SPECIALIST ROUTING ──────────────────────────────────────
Specialist    : {state['specialist']}
Reason        : {state['routing_reason']}

── CLINICAL EVIDENCE (RAG Retrieved) ───────────────────────
{evidence_preview}
{'='*60}
"""
    state['report'] = report
    return state


# ── Node 6: Create Alerts ──────────────────────────────────────────────────────
def create_alerts(state: TriageState) -> TriageState:
    """Generate tiered alert messages based on urgency."""
    print("[6/6] Creating alerts...")
    alerts = []
    urgency = state['urgency']
    specialist = state['specialist']
    patient_id = state['patient_id']

    if urgency == "CRITICAL":
        alerts.append(f"🚨 CRITICAL ALERT: Page {specialist} immediately for patient {patient_id}!")
        alerts.append(f"⏱️  SLA: Intervention required within 15 minutes.")
        alerts.append(f"📋 Reason: {state['urgency_reason']}")
    elif urgency == "URGENT":
        alerts.append(f"⚠️  URGENT: Notify {specialist} for patient {patient_id}.")
        alerts.append(f"⏱️  SLA: Evaluation required within 1 hour.")
        alerts.append(f"📋 Reason: {state['urgency_reason']}")
    elif urgency == "MODERATE":
        alerts.append(f"📅 MODERATE: Schedule {specialist} consult for patient {patient_id}.")
        alerts.append(f"⏱️  SLA: Same-day evaluation recommended.")
    else:
        alerts.append(f"✅ ROUTINE: No urgent findings for patient {patient_id}.")
        alerts.append(f"📅 Schedule routine follow-up within 1 week.")

    state['alerts'] = alerts
    return state


print("Agent nodes defined! ✅")

Agent nodes defined! ✅


## 9. Build LangGraph Workflow

In [ ]:
# Build the graph
workflow = StateGraph(TriageState)

# Add all 6 nodes
workflow.add_node("analyze_image",    analyze_image)
workflow.add_node("retrieve_evidence", retrieve_evidence)
workflow.add_node("classify_urgency", classify_urgency)
workflow.add_node("determine_routing", determine_routing)
workflow.add_node("generate_report",  generate_report)
workflow.add_node("create_alerts",    create_alerts)

# Linear chain
workflow.set_entry_point("analyze_image")
workflow.add_edge("analyze_image",    "retrieve_evidence")
workflow.add_edge("retrieve_evidence", "classify_urgency")
workflow.add_edge("classify_urgency", "determine_routing")
workflow.add_edge("determine_routing", "generate_report")
workflow.add_edge("generate_report",  "create_alerts")
workflow.add_edge("create_alerts",    END)

# Compile
triage_agent = workflow.compile()
print("LangGraph Triage Agent compiled! ✅")

LangGraph Triage Agent compiled! ✅


## 10. Helper Function to Run Agent

In [ ]:
def run_triage_agent(image_path, patient_id):
    """Run the full triage pipeline on a single image."""
    print(f"\n{'='*60}")
    print(f"RUNNING TRIAGE AGENT — Patient: {patient_id}")
    print(f"{'='*60}")

    initial_state = TriageState(
        image_path=image_path,
        patient_id=patient_id,
        predictions={},
        positive_findings=[],
        evidence="",
        urgency="",
        urgency_reason="",
        specialist="",
        routing_reason="",
        report="",
        alerts=[],
        timestamp=""
    )

    result = triage_agent.invoke(initial_state)

    print(result['report'])
    print("\n── ALERTS ──────────────────────────────────────────────")
    for alert in result['alerts']:
        print(alert)

    return result

print("Triage agent helper ready! ✅")

Triage agent helper ready! ✅


In [ ]:
import json, os

# Add your Kaggle credentials here
kaggle_credentials = {
    "username": "YOUR_KAGGLE_USERNAME",  # ← change this
    "key": "YOUR_KAGGLE_API_KEY"         # ← change this
}

!mkdir -p /root/.kaggle
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_credentials, f)
!chmod 600 /root/.kaggle/kaggle.json

# Download dataset
if not os.path.exists('/content/chexpert/train.csv'):
    print("Downloading CheXpert dataset...")
    !pip install kaggle -q
    !kaggle datasets download -d willarevalo/chexpert-v10-small -p /content/chexpert --unzip
    print("Dataset downloaded! ✅")
else:
    print("Dataset already exists! ✅")

Dataset URL: https://www.kaggle.com/datasets/willarevalo/chexpert-v10-small
License(s): unknown
chexpert-v10-small.zip: Skipping, found more recently modified local copy (use --force to force download)
Dataset downloaded! ✅


## 11. Test Case 1 — Pneumothorax Positive

In [ ]:
# Load CSV to find test cases
train_df = pd.read_csv(f"{DATA_DIR}train.csv")
frontal_df = train_df[train_df['Path'].str.contains('frontal')].copy()

# Fix image path
def get_image_path(row_path):
    return DATA_DIR + 'train/' + row_path.split('train/')[-1]

# Test Case 1: Pneumothorax positive
ptx_case = frontal_df[frontal_df['Pneumothorax'] == 1.0].iloc[0]
ptx_path = get_image_path(ptx_case['Path'])
print(f"Test image path: {ptx_path}")
print(f"File exists: {os.path.exists(ptx_path)}")

result1 = run_triage_agent(ptx_path, patient_id="PTX-001")


Test image path: /content/chexpert/CheXpert-v1.0-small/train/patient00005/study2/view1_frontal.jpg
File exists: True

RUNNING TRIAGE AGENT — Patient: PTX-001
[1/6] Analyzing image for patient PTX-001...

Ensemble Prediction Results
----------------------------------------
✓ Cardiomegaly: 33.4% (Negative)
✓ Pneumonia: 44.8% (Negative)
⚠️ Pneumothorax: 83.0% (POSITIVE)
✓ Edema: 13.3% (Negative)
⚠️ Pleural Effusion: 60.0% (POSITIVE)
[2/6] Retrieving clinical evidence for: ['Pneumothorax', 'Pleural Effusion']
[3/6] Classifying urgency...
    Urgency: CRITICAL
[4/6] Determining specialist routing...
    Routing to: Emergency Medicine + ICU
[5/6] Generating triage report...
[6/6] Creating alerts...

MEDICAL TRIAGE REPORT
Patient ID  : PTX-001
Timestamp   : 2026-03-22 03:26:04
Model       : Ensemble (DenseNet-121, EfficientNet-B0, ResNet-50, EfficientNet-B3)

── FINDINGS ────────────────────────────────────────────────
  • Cardiomegaly: 33.4%
  • Pneumonia: 44.8%
  • Pneumothorax: 83.0%
  • E

## 12. Test Case 2 — Cardiomegaly Positive

In [ ]:
# Test Case 2: Cardiomegaly positive
cardio_case = frontal_df[
    (frontal_df['Cardiomegaly'] == 1.0) &
    (frontal_df['Pneumothorax'] != 1.0)
].iloc[0]
cardio_path = DATA_DIR + 'train/' + cardio_case['Path'].split('train/')[-1]

result2 = run_triage_agent(cardio_path, patient_id="CARDIO-001")


RUNNING TRIAGE AGENT — Patient: CARDIO-001
[1/6] Analyzing image for patient CARDIO-001...

Ensemble Prediction Results
----------------------------------------
⚠️ Cardiomegaly: 78.0% (POSITIVE)
⚠️ Pneumonia: 68.3% (POSITIVE)
✓ Pneumothorax: 27.5% (Negative)
⚠️ Edema: 59.7% (POSITIVE)
✓ Pleural Effusion: 36.7% (Negative)
[2/6] Retrieving clinical evidence for: ['Cardiomegaly', 'Pneumonia', 'Edema']
[3/6] Classifying urgency...
    Urgency: URGENT
[4/6] Determining specialist routing...
    Routing to: Cardiology
[5/6] Generating triage report...
[6/6] Creating alerts...

MEDICAL TRIAGE REPORT
Patient ID  : CARDIO-001
Timestamp   : 2026-03-22 03:26:36
Model       : Ensemble (DenseNet-121, EfficientNet-B0, ResNet-50, EfficientNet-B3)

── FINDINGS ────────────────────────────────────────────────
  • Cardiomegaly: 78.0%
  • Pneumonia: 68.3%
  • Pneumothorax: 27.5%
  • Edema: 59.7%
  • Pleural Effusion: 36.7%

Positive Findings : Cardiomegaly, Pneumonia, Edema

── TRIAGE DECISION ─────────

## 13. Test Case 3 — Normal (No Findings)

In [ ]:
# Test Case 3: Normal case
normal_case = frontal_df[
    (frontal_df['Cardiomegaly'] != 1.0) &
    (frontal_df['Pneumonia'] != 1.0) &
    (frontal_df['Pneumothorax'] != 1.0) &
    (frontal_df['Edema'] != 1.0) &
    (frontal_df['Pleural Effusion'] != 1.0)
].iloc[0]
normal_path = DATA_DIR + 'train/' + normal_case['Path'].split('train/')[-1]

result3 = run_triage_agent(normal_path, patient_id="NORMAL-001")


RUNNING TRIAGE AGENT — Patient: NORMAL-001
[1/6] Analyzing image for patient NORMAL-001...

Ensemble Prediction Results
----------------------------------------
✓ Cardiomegaly: 22.1% (Negative)
⚠️ Pneumonia: 50.6% (POSITIVE)
✓ Pneumothorax: 31.5% (Negative)
✓ Edema: 22.8% (Negative)
✓ Pleural Effusion: 13.6% (Negative)
[2/6] Retrieving clinical evidence for: ['Pneumonia']
[3/6] Classifying urgency...
    Urgency: MODERATE
[4/6] Determining specialist routing...
    Routing to: Pulmonology / Infectious Disease
[5/6] Generating triage report...
[6/6] Creating alerts...

MEDICAL TRIAGE REPORT
Patient ID  : NORMAL-001
Timestamp   : 2026-03-22 03:26:40
Model       : Ensemble (DenseNet-121, EfficientNet-B0, ResNet-50, EfficientNet-B3)

── FINDINGS ────────────────────────────────────────────────
  • Cardiomegaly: 22.1%
  • Pneumonia: 50.6%
  • Pneumothorax: 31.5%
  • Edema: 22.8%
  • Pleural Effusion: 13.6%

Positive Findings : Pneumonia

── TRIAGE DECISION ─────────────────────────────────

## 14. Summary

In [ ]:
print("\n" + "="*60)
print("TRIAGE AGENT SUMMARY")
print("="*60)
for result, label in [(result1, 'PTX-001'), (result2, 'CARDIO-001'), (result3, 'NORMAL-001')]:
    print(f"Patient {label:12s} | Urgency: {result['urgency']:8s} | Specialist: {result['specialist']}")
print("="*60)
print("\n✅ RAG + Agentic Pipeline complete!")
print(f"Ensemble Models Used: DenseNet-121, EfficientNet-B0, ResNet-50, EfficientNet-B3")
print(f"Ensemble AUC: 0.8825 (above 0.85 baseline)")


TRIAGE AGENT SUMMARY
Patient PTX-001      | Urgency: CRITICAL | Specialist: Emergency Medicine + ICU
Patient CARDIO-001   | Urgency: URGENT   | Specialist: Cardiology
Patient NORMAL-001   | Urgency: MODERATE | Specialist: Pulmonology / Infectious Disease

✅ RAG + Agentic Pipeline complete!
Ensemble Models Used: DenseNet-121, EfficientNet-B0, ResNet-50, EfficientNet-B3
Ensemble AUC: 0.8825 (above 0.85 baseline)
